# 03 — News Ticker Mapping and Export

## Purpose
Take the article-level sentiment dataset (one row per article) and produce the ticker-level news tables, keeping **only the 10 project tickers**.

Example: an article tagged `[AAPL, COST, BAC, BABA, AMZN]` becomes two rows, `AAPL` and `AMZN`. `COST`, `BAC` and `BABA` are discarded.

## Input
`data/processed/news_articles_sentiment.csv` (from `02_news_cleaning_and_finbert`). Its `symbols` column holds the **full** Alpaca ticker list, which is what this notebook filters.

## Outputs
| File | Grain | Contents |
|---|---|---|
| `news_tickers.csv` (**v2**) | one row per `(article_id, ticker)` | article ID, ticker, `n_symbols_total`, `n_project_tickers`, `published_at` (UTC) and **`reaction_date`** (the warehouse association table, now linked to trading days) |
| `news_clean.csv` | one row per `(article_id, ticker)` | the pair plus article metadata and FinBERT sentiment (flattened export for analysis / Power BI) |

## What is new in version 2: `reaction_date`
Prices exist only on trading days, and the market reacts to news in New York time. So every article gets a `reaction_date`, the first trading day on which the market could react:
- `published_at` (UTC) is converted to `America/New_York` (daylight saving handled automatically);
- published **before 16:00** on a trading day -> that same day;
- published **at or after 16:00**, on a **weekend** or on an **NYSE holiday** -> the next trading day.

Examples: Friday 17:30 -> Monday; Saturday 10:00 -> Monday; Tuesday 10:00 -> Tuesday. The article text is not changed and FinBERT is not re-run. The date is joined to prices later (in the warehouse) through `(ticker, reaction_date)`.

## Important interpretation
FinBERT is **not** re-run here. The article's sentiment is copied to every ticker it mentions, so it describes the article text, not sentiment toward each specific company. Two columns help with this:
- `n_project_tickers`: how many of the 10 project tickers the article mentions.
- `n_symbols_total`: how many tickers Alpaca tagged in total. Articles tagged with dozens of tickers are usually market roundups, so consider down-weighting or excluding them in the analysis.

In [1]:
import ast
import json
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Paths (same ../../data convention as notebook 02) ----
PROCESSED_DIR = Path("../../data/processed")
INPUT_PATH = PROCESSED_DIR / "news_articles_sentiment.csv"
TICKERS_PATH = PROCESSED_DIR / "news_tickers.csv"
FLAT_PATH = PROCESSED_DIR / "news_clean.csv"
MARKET_PATH = PROCESSED_DIR / "market_clean.csv"   # only used to cross-check the trading calendar

# ---- Canonical project tickers ----
TICKERS = ["AAPL", "JPM", "MSFT", "NVDA", "JNJ", "XOM", "KO", "AMZN", "LLY", "TSLA"]
PROJECT_TICKERS = set(TICKERS)
print("Project tickers:", sorted(PROJECT_TICKERS))

Project tickers: ['AAPL', 'AMZN', 'JNJ', 'JPM', 'KO', 'LLY', 'MSFT', 'NVDA', 'TSLA', 'XOM']


## 1. Load only the columns needed

`sentiment_text` is the largest column and is not needed here, so it is skipped. IDs are read as text so they are never converted to numbers.

In [2]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Could not find {INPUT_PATH.resolve()}. Run notebook 02 first.")

WANTED = [
    "article_id", "published_at", "title", "summary", "source", "url", "symbols",
    "finbert_label", "finbert_positive", "finbert_negative", "finbert_neutral", "finbert_score",
]
articles = pd.read_csv(
    INPUT_PATH,
    usecols=lambda c: c in WANTED,
    dtype={"article_id": str, "symbols": str},
)
print(f"Loaded {len(articles):,} article-level rows")

required = {"article_id", "published_at", "title", "symbols", "finbert_label", "finbert_score"}
missing = required - set(articles.columns)
if missing:
    raise ValueError(f"Input is missing required columns: {sorted(missing)}")

# Optional columns: create empty ones so the export has a stable schema.
for col in WANTED:
    if col not in articles.columns:
        articles[col] = pd.NA

articles["article_id"] = articles["article_id"].str.strip()
if articles["article_id"].duplicated().any():
    raise ValueError("Input must have one row per unique article_id.")

articles["published_at"] = pd.to_datetime(articles["published_at"], utc=True, errors="coerce")
articles["finbert_score"] = pd.to_numeric(articles["finbert_score"], errors="coerce")

Loaded 79,549 article-level rows


## 2. Parse the symbols and keep only project tickers

The `symbols` column stores the full ticker list as JSON (written by notebook 02). The parser also accepts Python-list-like strings and plain comma-separated text. Tickers are upper-cased and de-duplicated, then filtered to `PROJECT_TICKERS`.

In [3]:
def parse_symbols(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    if isinstance(value, (list, tuple, set)):
        items = list(value)
    elif isinstance(value, str):
        value = value.strip()
        if not value:
            return []
        try:
            parsed = json.loads(value)
        except (json.JSONDecodeError, TypeError):
            try:
                parsed = ast.literal_eval(value)
            except (ValueError, SyntaxError):
                parsed = value.split(",")
        items = list(parsed) if isinstance(parsed, (list, tuple, set)) else [parsed]
    else:
        return []
    return list(dict.fromkeys(
        str(item).strip().upper() for item in items
        if item is not None and str(item).strip()
    ))


def keep_project_tickers(all_symbols):
    return sorted(t for t in all_symbols if t in PROJECT_TICKERS)


# Sanity check of the filtering rule, using the example from the project notes.
assert keep_project_tickers(parse_symbols('["AAPL", "COST", "BAC", "BABA", "AMZN"]')) == ["AAPL", "AMZN"]
assert keep_project_tickers(parse_symbols("tsla, nvda")) == ["NVDA", "TSLA"]
assert keep_project_tickers(parse_symbols("[]")) == []

all_symbols = articles["symbols"].apply(parse_symbols)
articles["n_symbols_total"] = all_symbols.apply(len)
articles["project_tickers"] = all_symbols.apply(keep_project_tickers)
articles["n_project_tickers"] = articles["project_tickers"].apply(len)
del all_symbols

has_project = articles["n_project_tickers"].gt(0)
print("Articles in input:                    ", f"{len(articles):,}")
print("Articles with no symbols at all:      ", f"{int(articles['n_symbols_total'].eq(0).sum()):,}")
print("Articles with >=1 project ticker:     ", f"{int(has_project.sum()):,}")
print("Articles with no project ticker (out):", f"{int((~has_project).sum()):,}")
print("Articles mentioning 2+ project tickers:", f"{int(articles['n_project_tickers'].gt(1).sum()):,}")

Articles in input:                     79,549
Articles with no symbols at all:       0
Articles with >=1 project ticker:      79,549
Articles with no project ticker (out): 0
Articles mentioning 2+ project tickers: 14,439


## 3. Build the article–ticker pairs

Only articles with at least one project ticker appear here. Out-of-scope tickers are dropped; the article itself stays in `news_articles_sentiment.csv`. The file is saved in section 3c, after the reaction date is added.

In [4]:
mapping = (
    articles.loc[has_project, ["article_id", "project_tickers"]]
    .explode("project_tickers")
    .rename(columns={"project_tickers": "ticker"})
    .drop_duplicates(subset=["article_id", "ticker"])
    .sort_values(["article_id", "ticker"])
    .reset_index(drop=True)
)

assert not mapping.duplicated(["article_id", "ticker"]).any()
assert mapping["ticker"].isin(PROJECT_TICKERS).all()
assert len(mapping) == int(articles.loc[has_project, "n_project_tickers"].sum())

print(f"{len(mapping):,} article-ticker pairs built (saved in section 3c)")
display(mapping.head(10))

102,767 article-ticker pairs built (saved in section 3c)


,article_id,ticker
0,10903978,KO
1,10978750,AAPL
2,11007650,AAPL
3,11013619,XOM
4,11093651,AAPL
5,11118958,AAPL
6,11118958,MSFT
7,11159875,AAPL
8,11159875,AMZN
9,11159875,MSFT


## 3a. NYSE trading calendar

Weekends plus the NYSE full-day holidays (including the 2025-01-09 market closure for the National Day of Mourning). It is the same list as in `05_events_cleaning.ipynb`, extended to the end of 2026 so that articles published on the evening of 2025-12-31 roll to 2026-01-02. If `market_clean.csv` exists, the calendar is cross-checked against the real price dates.

In [5]:
NYSE_HOLIDAYS = pd.to_datetime([
    # 2021
    "2021-01-01", "2021-01-18", "2021-02-15", "2021-04-02", "2021-05-31", "2021-07-05", "2021-09-06", "2021-11-25", "2021-12-24",
    # 2022
    "2022-01-17", "2022-02-21", "2022-04-15", "2022-05-30", "2022-06-20", "2022-07-04", "2022-09-05", "2022-11-24", "2022-12-26",
    # 2023
    "2023-01-02", "2023-01-16", "2023-02-20", "2023-04-07", "2023-05-29", "2023-06-19", "2023-07-04", "2023-09-04", "2023-11-23", "2023-12-25",
    # 2024
    "2024-01-01", "2024-01-15", "2024-02-19", "2024-03-29", "2024-05-27", "2024-06-19", "2024-07-04", "2024-09-02", "2024-11-28", "2024-12-25",
    # 2025
    "2025-01-01", "2025-01-09", "2025-01-20", "2025-02-17", "2025-04-18", "2025-05-26", "2025-06-19", "2025-07-04", "2025-09-01", "2025-11-27", "2025-12-25",
    # 2026
    "2026-01-01", "2026-01-19", "2026-02-16", "2026-04-03", "2026-05-25", "2026-06-19", "2026-07-03", "2026-09-07", "2026-11-26", "2026-12-25",
])
HOLIDAY_SET = set(NYSE_HOLIDAYS)

_all_days = pd.date_range("2020-12-01", "2026-12-31", freq="D")
TRADING_DAYS = pd.DatetimeIndex([d for d in _all_days if d.dayofweek < 5 and d not in HOLIDAY_SET]).as_unit("ns")
print(f"Calendar: {len(TRADING_DAYS):,} trading days from {TRADING_DAYS.min().date()} to {TRADING_DAYS.max().date()}")

LAST_PRICE_DAY = pd.Timestamp("2025-12-30")     # last day in market_clean.csv (documented data cutoff)
if MARKET_PATH.exists():
    price_days = pd.DatetimeIndex(pd.to_datetime(pd.read_csv(MARKET_PATH, usecols=["date"])["date"].drop_duplicates()))
    LAST_PRICE_DAY = price_days.max()
    span = TRADING_DAYS[(TRADING_DAYS >= price_days.min()) & (TRADING_DAYS <= price_days.max())]
    only_calendar = span.difference(price_days)
    only_prices = price_days.difference(span)
    print(f"Calendar vs market_clean.csv: {len(only_calendar)} days only in calendar, {len(only_prices)} days only in prices")
    assert len(only_calendar) == 0 and len(only_prices) == 0, "calendar does not match the price dates"
else:
    print("market_clean.csv not found: calendar not cross-checked.")

Calendar: 1,529 trading days from 2020-12-01 to 2026-12-31
Calendar vs market_clean.csv: 0 days only in calendar, 0 days only in prices


## 3b. Reaction date

The rule is applied to every article once (not per ticker). It is vectorised: the New York publication time is moved forward by one day when the hour is 16 or later, and the result is rolled forward to the first trading day on or after it.

In [6]:
def to_reaction_date(published_utc):
    # UTC timestamps (tz-aware Series) -> first trading day the market could react (naive Series)
    ny = published_utc.dt.tz_convert("America/New_York")
    local_day = ny.dt.tz_localize(None).dt.normalize()
    after_close = (ny.dt.hour >= 16).fillna(False).astype(int)
    base = (local_day + pd.to_timedelta(after_close, unit="D")).dt.as_unit("ns")
    out = pd.Series(pd.NaT, index=published_utc.index, dtype="datetime64[ns]")
    ok = base.notna()
    pos = TRADING_DAYS.searchsorted(base[ok].to_numpy(), side="left")
    if (pos >= len(TRADING_DAYS)).any():
        raise ValueError("Calendar ends before some articles; extend _all_days.")
    out.loc[ok] = TRADING_DAYS[pos].to_numpy()
    return out


def _check(utc_text, expected_day):
    got = to_reaction_date(pd.Series(pd.to_datetime([utc_text], utc=True))).iloc[0]
    assert got == pd.Timestamp(expected_day), (utc_text, got, expected_day)


# Hand-checked examples (New York time in brackets)
_check("2025-03-07T22:30:00Z", "2025-03-10")   # Friday 17:30 EST        -> Monday
_check("2025-03-08T15:00:00Z", "2025-03-10")   # Saturday 10:00 EST      -> Monday
_check("2025-03-04T15:00:00Z", "2025-03-04")   # Tuesday 10:00 EST       -> Tuesday (same day)
_check("2025-06-11T19:59:00Z", "2025-06-11")   # Wednesday 15:59 EDT     -> same day
_check("2025-06-11T20:00:00Z", "2025-06-12")   # Wednesday 16:00 EDT     -> next day (at the close)
_check("2025-01-08T22:30:00Z", "2025-01-10")   # Wed after close; 2025-01-09 market closed -> Friday
_check("2024-01-01T15:00:00Z", "2024-01-02")   # New Year's Day (holiday) -> Tuesday
_check("2025-12-31T22:30:00Z", "2026-01-02")   # Wed after close; 2026-01-01 holiday -> Friday
_check("2021-03-15T20:30:00Z", "2021-03-16")   # 16:30 EDT just after the US clock change (Mar 14)
print("Reaction-date self-tests passed.")

articles["reaction_date"] = to_reaction_date(articles["published_at"])
ny_day = articles["published_at"].dt.tz_convert("America/New_York").dt.tz_localize(None).dt.normalize()
articles["_rolled"] = articles["reaction_date"].ne(ny_day) & articles["reaction_date"].notna()

print("Articles with a reaction date:        ", f"{int(articles['reaction_date'].notna().sum()):,}")
print("Articles without (missing timestamp): ", f"{int(articles['reaction_date'].isna().sum()):,}")
print("Share moved to a later day:           ", f"{articles['_rolled'].mean():.1%}")
print("Reaction date after the last price day:", f"{int((articles['reaction_date'] > LAST_PRICE_DAY).sum()):,}", "(no price to compare with; expected for the last days of 2025)")
display(articles[["article_id", "published_at", "reaction_date"]].head(8))

Reaction-date self-tests passed.
Articles with a reaction date:         79,549
Articles without (missing timestamp):  0
Share moved to a later day:            18.9%
Reaction date after the last price day: 40 (no price to compare with; expected for the last days of 2025)


,article_id,published_at,reaction_date
0,10903978,2022-12-12 14:11:17+00:00,2022-12-12
1,10978750,2022-01-03 14:03:00+00:00,2022-01-03
2,11007650,2023-01-09 15:46:00+00:00,2023-01-09
3,11013619,2022-01-10 14:19:00+00:00,2022-01-10
4,11093651,2021-01-29 14:04:00+00:00,2021-01-29
5,11118958,2021-02-01 13:53:00+00:00,2021-02-01
6,11159875,2022-02-08 14:32:00+00:00,2022-02-08
7,11374269,2022-03-16 13:06:00+00:00,2022-03-16


## 3c. Save `news_tickers.csv` (version 2)

Adds the article-level columns to every article–ticker pair. If a previous `news_tickers.csv` exists, the set of pairs must be identical: only columns are added, no pair is added or removed.

In [8]:
tickers_out = mapping.merge(
    articles[["article_id", "n_symbols_total", "n_project_tickers", "published_at", "reaction_date"]],
    on="article_id", how="left", validate="many_to_one",
).sort_values(["article_id", "ticker"]).reset_index(drop=True)

# ---- Checks ----
assert len(tickers_out) == len(mapping), "Join changed the number of rows."
assert not tickers_out.duplicated(["article_id", "ticker"]).any()
assert tickers_out["ticker"].isin(PROJECT_TICKERS).all()
dated = tickers_out["reaction_date"].notna()
assert (dated == tickers_out["published_at"].notna()).all(), "reaction_date missing where published_at exists"
assert tickers_out.loc[dated, "reaction_date"].isin(TRADING_DAYS).all(), "reaction_date is not a trading day"
local_day = tickers_out["published_at"].dt.tz_convert("America/New_York").dt.tz_localize(None).dt.normalize()
assert (tickers_out.loc[dated, "reaction_date"] >= local_day[dated]).all(), "reaction_date earlier than the New York date"

if TICKERS_PATH.exists():
    old = pd.read_csv(TICKERS_PATH, usecols=["article_id", "ticker"], dtype=str)
    same = set(map(tuple, old.values.tolist())) == set(map(tuple, tickers_out[["article_id", "ticker"]].values.tolist()))
    print(f"Previous file: {len(old):,} pairs; identical set of pairs: {same}")
    assert same, "The set of article-ticker pairs changed. Only columns should be added."

export = tickers_out[["article_id", "ticker", "n_symbols_total", "n_project_tickers", "published_at", "reaction_date"]].copy()
export["reaction_date"] = export["reaction_date"].dt.strftime("%Y-%m-%d")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
export.to_csv(TICKERS_PATH, index=False)
print(f"Saved {len(export):,} article-ticker rows to {TICKERS_PATH.resolve()}")
display(export.head(10))

Saved 102,767 article-ticker rows to D:\DEPI\MarketPulse Graduation Project\data\processed\news_tickers.csv


,article_id,ticker,n_symbols_total,n_project_tickers,published_at,reaction_date
0,10903978,KO,5,1,2022-12-12 14:11:17+00:00,2022-12-12
1,10978750,AAPL,1,1,2022-01-03 14:03:00+00:00,2022-01-03
2,11007650,AAPL,3,1,2023-01-09 15:46:00+00:00,2023-01-09
3,11013619,XOM,3,1,2022-01-10 14:19:00+00:00,2022-01-10
4,11093651,AAPL,4,1,2021-01-29 14:04:00+00:00,2021-01-29
5,11118958,AAPL,4,2,2021-02-01 13:53:00+00:00,2021-02-01
6,11118958,MSFT,4,2,2021-02-01 13:53:00+00:00,2021-02-01
7,11159875,AAPL,6,3,2022-02-08 14:32:00+00:00,2022-02-08
8,11159875,AMZN,6,3,2022-02-08 14:32:00+00:00,2022-02-08
9,11159875,MSFT,6,3,2022-02-08 14:32:00+00:00,2022-02-08


## 4. Build `news_clean.csv` (flattened: one row per article–ticker pair)

Joins article metadata and sentiment onto each pair. `news_id` is a generated row ID; `article_id` is the original Alpaca ID. `date` is the **UTC calendar date** of publication; the market-reaction date (after close, weekends, holidays) is a separate later derivation.

In [9]:
article_columns = [
    "article_id", "published_at", "title", "summary", "source", "url",
    "n_project_tickers", "n_symbols_total",
    "finbert_label", "finbert_positive", "finbert_negative", "finbert_neutral", "finbert_score",
]

flat = mapping.merge(
    articles[article_columns], on="article_id", how="left", validate="many_to_one"
)
flat["date"] = flat["published_at"].dt.strftime("%Y-%m-%d")

flat = flat.sort_values(["article_id", "ticker"]).reset_index(drop=True)
flat.insert(0, "news_id", range(1, len(flat) + 1))

flat = flat[[
    "news_id", "article_id", "ticker", "published_at", "date", "title", "summary", "source", "url",
    "n_project_tickers", "n_symbols_total",
    "finbert_label", "finbert_positive", "finbert_negative", "finbert_neutral", "finbert_score",
]]

# ---- Integrity checks ----
assert len(flat) == len(mapping), "Join changed the number of rows."
assert not flat.duplicated(["article_id", "ticker"]).any()
assert flat["news_id"].is_unique
assert flat["ticker"].isin(PROJECT_TICKERS).all()

flat.to_csv(FLAT_PATH, index=False)
print(f"Saved {len(flat):,} rows to {FLAT_PATH.resolve()}")
display(flat.head(10))

Saved 102,767 rows to D:\DEPI\MarketPulse Graduation Project\data\processed\news_clean.csv


,news_id,article_id,ticker,published_at,date,title,summary,source,url,n_project_tickers,n_symbols_total,finbert_label,finbert_positive,finbert_negative,finbert_neutral,finbert_score
0,1,10903978,KO,2022-12-12 14:11:17+00:00,2022-12-12,This Day In Market History: Warren Buffett Buy...,"Each day, Benzinga will take a look back at a ...",benzinga,https://www.benzinga.com/general/education/17/...,1,5,neutral,0.039130,0.028487,0.932384,0.010643
1,2,10978750,AAPL,2022-01-03 14:03:00+00:00,2022-01-03,This Day In Market History: Jobs And Wozniak I...,"Each day, Benzinga takes a look back at a nota...",NaN,https://www.benzinga.com/general/education/21/...,1,1,neutral,0.022782,0.037532,0.939685,-0.014750
2,3,11007650,AAPL,2023-01-09 15:46:00+00:00,2023-01-09,This Day In Market History: Apple Introduces T...,"Each day, Benzinga takes a look back at a nota...",benzinga,https://www.benzinga.com/general/education/20/...,1,3,neutral,0.048113,0.024673,0.927214,0.023441
3,4,11013619,XOM,2022-01-10 14:19:00+00:00,2022-01-10,This Day In Market History: Rockefeller Founds...,"Each day, Benzinga takes a look back at a nota...",benzinga,https://www.benzinga.com/general/education/22/...,1,3,neutral,0.024851,0.056945,0.918204,-0.032094
4,5,11093651,AAPL,2021-01-29 14:04:00+00:00,2021-01-29,This Day In Market History: US Treasury Introd...,"Each day, Benzinga takes a look back at a nota...",NaN,https://www.benzinga.com/markets/bonds/21/01/1...,1,4,neutral,0.059405,0.019906,0.920689,0.039499
5,6,11118958,AAPL,2021-02-01 13:53:00+00:00,2021-02-01,This Day In Market History: Alphabet Briefly B...,"Each day, Benzinga takes a look back at a nota...",NaN,https://www.benzinga.com/markets/cryptocurrenc...,2,4,neutral,0.184619,0.014462,0.800919,0.170157
6,7,11118958,MSFT,2021-02-01 13:53:00+00:00,2021-02-01,This Day In Market History: Alphabet Briefly B...,"Each day, Benzinga takes a look back at a nota...",NaN,https://www.benzinga.com/markets/cryptocurrenc...,2,4,neutral,0.184619,0.014462,0.800919,0.170157
7,8,11159875,AAPL,2022-02-08 14:32:00+00:00,2022-02-08,This Day In Market History: Nasdaq Opens For B...,"Each day, Benzinga takes a look back at a nota...",benzinga,https://www.benzinga.com/general/education/22/...,3,6,neutral,0.026799,0.031563,0.941638,-0.004765
8,9,11159875,AMZN,2022-02-08 14:32:00+00:00,2022-02-08,This Day In Market History: Nasdaq Opens For B...,"Each day, Benzinga takes a look back at a nota...",benzinga,https://www.benzinga.com/general/education/22/...,3,6,neutral,0.026799,0.031563,0.941638,-0.004765
9,10,11159875,MSFT,2022-02-08 14:32:00+00:00,2022-02-08,This Day In Market History: Nasdaq Opens For B...,"Each day, Benzinga takes a look back at a nota...",benzinga,https://www.benzinga.com/general/education/22/...,3,6,neutral,0.026799,0.031563,0.941638,-0.004765


## 5. Quality checks

A project ticker with zero articles, or a year with very few, points to a collection gap upstream (the earlier raw CSV had many empty months), not a mapping bug.

In [10]:
summary = {
    "unique_articles_in_input": int(len(articles)),
    "articles_with_project_ticker": int(mapping["article_id"].nunique()),
    "articles_dropped_no_project_ticker": int((~has_project).sum()),
    "article_ticker_pairs": int(len(mapping)),
    "flattened_rows": int(len(flat)),
    "missing_sentiment_label_rows": int(flat["finbert_label"].isna().sum()),
    "missing_sentiment_score_rows": int(flat["finbert_score"].isna().sum()),
    "missing_published_at_rows": int(flat["published_at"].isna().sum()),
}
display(pd.DataFrame([summary]).T.rename(columns={0: "value"}))

per_ticker = (
    mapping["ticker"].value_counts().reindex(TICKERS, fill_value=0)
    .rename_axis("ticker").to_frame("article_count")
)
display(per_ticker)
if per_ticker["article_count"].eq(0).any():
    print("WARNING: tickers with zero articles:", per_ticker.index[per_ticker["article_count"].eq(0)].tolist())

year = flat["published_at"].dt.year
display(pd.crosstab(flat["ticker"], year).reindex(TICKERS, fill_value=0))

,value
unique_articles_in_input,79549
articles_with_project_ticker,79549
articles_dropped_no_project_ticker,0
article_ticker_pairs,102767
flattened_rows,102767
missing_sentiment_label_rows,0
missing_sentiment_score_rows,0
missing_published_at_rows,0


,article_count
ticker,
AAPL,19354
JPM,3646
MSFT,13346
NVDA,14754
JNJ,3320
XOM,2144
KO,1259
AMZN,14906
LLY,3142


published_at,2021,2022,2023,2024,2025
ticker,,,,,
AAPL,3645,4246,4272,3873,3318
JPM,465,705,921,704,851
MSFT,1600,1937,3568,3151,3090
NVDA,1036,1632,2719,4401,4966
JNJ,1161,575,574,500,510
XOM,289,508,523,449,375
KO,254,285,253,190,277
AMZN,2750,3212,2978,2815,3151
LLY,477,442,597,856,770


## Completion checklist

- [ ] `TICKERS` matches the team's canonical company list.
- [ ] `news_tickers.csv` has one row per unique `(article_id, ticker)`, project tickers only, with `reaction_date` filled for every article that has a timestamp.
- [ ] The reaction-date self-tests and the calendar cross-check passed.
- [ ] `news_clean.csv` has the same rows plus article metadata and sentiment.
- [ ] FinBERT was not re-run here.
- [ ] Sentiment is documented as article-level, not company-specific.
- [ ] Every project ticker has articles in every year (otherwise investigate upstream collection).
- [ ] Person 4 has the final column names: `article_id, ticker, n_symbols_total, n_project_tickers, published_at, reaction_date`.